# EA-TTM — M5 Data Contract and Temporal Split Audit

**Project:** Extreme-Aware Tiny Time Mixers (EA-TTM) for Retail Demand Forecasting under Regime Shifts  
**Phase:** 02 — M5 data contract, integrity, preprocessing, and temporal split audit  
**Scope:** Validate the exact retail dataset used by the project, establish immutable raw-data fingerprints, convert validated source files to efficient Parquet artifacts, and freeze the chronological train/validation/test protocol.

This phase performs **no model training** and does **not** inspect the locked final test targets for model selection.

## Phase 01 dependency

The model contract established in Phase 01 is:

- context length: $L=512$ daily observations;
- effective forecast horizon: $H=28$;
- selected TTM revision: `512-48-ft-r2.1`;
- native TTM prediction length: $48$ with a 28-step prediction filter;
- Slim Decoder output:

$$
\mathbf{D}
\in
\mathbb{R}^{B\times C\times 9\times 128}.
$$

The dataset contract defined here must therefore provide at least $L+H=540$ chronologically ordered observations for every evaluated store-item series.


## M5 source contract

The project uses the **M5 Forecasting — Accuracy** retail dataset at the bottom store-item level.

Three source files are required:

- `sales_train_evaluation.csv`;
- `calendar.csv`;
- `sell_prices.csv`.

The sales matrix is expected to contain one row per bottom-level store-item series and daily demand columns

$$
d_1,d_2,\ldots,d_{1941}.
$$

No random split is permitted. All model development and evaluation must preserve temporal order.


## Pre-registered temporal protocol

Let $o$ denote the final observed day at a forecast origin. For context length $L=512$ and horizon $H=28$, a forecasting sample is:

$$
\mathbf{x}^{(o)}
=
[y_{o-L+1},\ldots,y_o]
$$

with target:

$$
\mathbf{y}^{(o)}
=
[y_{o+1},\ldots,y_{o+H}].
$$

The approved origins are:

$$
\mathcal{O}_{train}
=
\{512,\ 960,\ 1409,\ 1857\},
$$

$$
o_{val}=1885,
\qquad
o_{test}=1913.
$$

Therefore:

- training targets end at $d_{1885}$;
- validation is $d_{1886}:d_{1913}$;
- the locked final test is $d_{1914}:d_{1941}$.

The final test interval is reserved for the frozen final configuration and must not be used for architecture or hyperparameter selection.


In [1]:
from __future__ import annotations

import hashlib
import json
import os
import platform
import sys
from datetime import datetime, timezone
from pathlib import Path
from typing import Any

import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq


In [2]:
from google.colab import drive

drive.mount("/content/drive")

PROJECT_ROOT = Path("/content/drive/MyDrive/EA_TTM_RETAIL")

RAW_M5_DIR = PROJECT_ROOT / "data" / "raw" / "m5"
PROCESSED_M5_DIR = PROJECT_ROOT / "data" / "processed" / "m5"
METADATA_DIR = PROJECT_ROOT / "metadata"
RESULTS_DIR = PROJECT_ROOT / "results" / "tables"

for directory in (RAW_M5_DIR, PROCESSED_M5_DIR, METADATA_DIR, RESULTS_DIR):
    directory.mkdir(parents=True, exist_ok=True)

REQUIRED_RAW_FILES = {
    "sales": RAW_M5_DIR / "sales_train_evaluation.csv",
    "calendar": RAW_M5_DIR / "calendar.csv",
    "prices": RAW_M5_DIR / "sell_prices.csv",
}

missing_raw_files = [
    str(path)
    for path in REQUIRED_RAW_FILES.values()
    if not path.exists()
]

assert not missing_raw_files, (
    "Required M5 source files are missing from the project raw-data directory: "
    + "; ".join(missing_raw_files)
)


Mounted at /content/drive


## Immutable raw-data fingerprints

Each source file is hashed with SHA-256. The project dataset fingerprint is derived from the ordered set of source-file hashes:

$$
F_{data}
=
\operatorname{SHA256}
\left(
F_{sales}
\Vert
F_{calendar}
\Vert
F_{prices}
\right).
$$

Any later change to a raw file therefore produces a different dataset identity.


In [3]:
def sha256_file(path: Path, chunk_size: int = 8 * 1024 * 1024) -> str:
    """Compute a streaming SHA-256 digest without loading the file into memory."""
    digest = hashlib.sha256()

    with path.open("rb") as file_handle:
        while chunk := file_handle.read(chunk_size):
            digest.update(chunk)

    return digest.hexdigest()


raw_file_manifest: dict[str, dict[str, Any]] = {}

for logical_name, path in REQUIRED_RAW_FILES.items():
    raw_file_manifest[logical_name] = {
        "path": str(path),
        "size_bytes": int(path.stat().st_size),
        "sha256": sha256_file(path),
    }

combined_fingerprint_payload = json.dumps(
    raw_file_manifest,
    sort_keys=True,
    separators=(",", ":"),
).encode("utf-8")

dataset_sha256 = hashlib.sha256(combined_fingerprint_payload).hexdigest()

print("Dataset SHA-256:", dataset_sha256)


Dataset SHA-256: 83962ce8bbcb0757215d045f81cac3a7cf4465943334ffa168af326bedc5ed59


## Schema validation

The audit validates the structural contract before deriving any processed artifact.

For the sales table:

- exactly 30,490 bottom-level store-item series are expected;
- identifier columns must be present;
- the daily demand sequence must be exactly $d_1$ through $d_{1941}$.

For the calendar and price tables, all fields required by the approved architecture are explicitly checked before use.


In [4]:
SALES_IDENTIFIER_COLUMNS = [
    "id",
    "item_id",
    "dept_id",
    "cat_id",
    "store_id",
    "state_id",
]

EXPECTED_DAY_COLUMNS = [f"d_{day}" for day in range(1, 1942)]

CALENDAR_REQUIRED_COLUMNS = [
    "date",
    "wm_yr_wk",
    "weekday",
    "wday",
    "month",
    "year",
    "d",
    "event_name_1",
    "event_type_1",
    "event_name_2",
    "event_type_2",
    "snap_CA",
    "snap_TX",
    "snap_WI",
]

PRICE_REQUIRED_COLUMNS = [
    "store_id",
    "item_id",
    "wm_yr_wk",
    "sell_price",
]

sales_header = pd.read_csv(REQUIRED_RAW_FILES["sales"], nrows=0).columns.tolist()
calendar_header = pd.read_csv(REQUIRED_RAW_FILES["calendar"], nrows=0).columns.tolist()
prices_header = pd.read_csv(REQUIRED_RAW_FILES["prices"], nrows=0).columns.tolist()

observed_day_columns = [
    column
    for column in sales_header
    if column.startswith("d_")
]

assert all(column in sales_header for column in SALES_IDENTIFIER_COLUMNS)
assert observed_day_columns == EXPECTED_DAY_COLUMNS, (
    "Sales day-column contract does not match d_1 through d_1941 exactly."
)
assert all(column in calendar_header for column in CALENDAR_REQUIRED_COLUMNS)
assert all(column in prices_header for column in PRICE_REQUIRED_COLUMNS)


## Memory-aware source loading

The daily sales values are loaded as signed 32-bit integers, which is sufficient for observed M5 unit-sales counts while avoiding the memory overhead of 64-bit integer storage.

Identifier columns use Pandas string dtype. The calendar and price files retain their explicit semantic fields for later covariate construction.


In [5]:
sales_dtypes = {
    column: "int32"
    for column in EXPECTED_DAY_COLUMNS
}

sales_dtypes.update(
    {
        column: "string"
        for column in SALES_IDENTIFIER_COLUMNS
    }
)

sales = pd.read_csv(
    REQUIRED_RAW_FILES["sales"],
    dtype=sales_dtypes,
)

calendar = pd.read_csv(
    REQUIRED_RAW_FILES["calendar"],
    parse_dates=["date"],
)

prices = pd.read_csv(
    REQUIRED_RAW_FILES["prices"],
    dtype={
        "store_id": "string",
        "item_id": "string",
        "wm_yr_wk": "int32",
        "sell_price": "float32",
    },
)

assert sales.shape[0] == 30_490, (
    f"Expected 30,490 bottom-level series; observed {sales.shape[0]:,}."
)
assert not sales["id"].duplicated().any(), "Duplicate series IDs were detected."
assert sales[EXPECTED_DAY_COLUMNS].isna().sum().sum() == 0, (
    "Missing demand observations were detected in the sales matrix."
)
assert (sales[EXPECTED_DAY_COLUMNS] < 0).sum().sum() == 0, (
    "Negative demand values were detected."
)

print("Sales shape:", sales.shape)
print("Calendar shape:", calendar.shape)
print("Prices shape:", prices.shape)


Sales shape: (30490, 1947)
Calendar shape: (1969, 14)
Prices shape: (6841121, 4)


## Calendar continuity and forecast-date contract

The sales-day labels must map one-to-one to calendar dates. For each $d_i$ used by the project:

$$
d_i
\longleftrightarrow
t_i.
$$

M5 day identifiers such as `d_1`, `d_10`, and `d_100` are strings. A direct string sort would be **lexicographic rather than chronological**. The audit therefore extracts the numeric day index

$$
i = \operatorname{int}\!\left(d_i[2:]\right)
$$

and verifies the exact sequence

$$
1,2,\ldots,1941
$$

before checking that the corresponding calendar dates increase monotonically.


In [6]:
calendar_day_counts = calendar["d"].value_counts(dropna=False)

required_calendar_days = set(EXPECTED_DAY_COLUMNS)
available_calendar_days = set(calendar["d"].dropna().astype(str))

assert required_calendar_days.issubset(available_calendar_days), (
    "Calendar file does not cover all observed sales days."
)

duplicated_required_calendar_days = [
    day
    for day in EXPECTED_DAY_COLUMNS
    if int(calendar_day_counts.get(day, 0)) != 1
]

assert not duplicated_required_calendar_days, (
    "Sales-day/calendar mapping is not one-to-one for: "
    + ", ".join(duplicated_required_calendar_days[:10])
)

# IMPORTANT:
# M5 day labels are strings such as d_1, d_2, ..., d_1941.
# Sorting these labels directly as strings is lexicographic
# (d_1, d_10, d_100, ...), not chronological.
# We therefore derive the numeric day index explicitly and sort on it.
calendar_lookup = (
    calendar.loc[
        calendar["d"].isin(EXPECTED_DAY_COLUMNS),
        ["d", "date", "wm_yr_wk"],
    ]
    .drop_duplicates("d")
    .assign(
        day_index=lambda frame: (
            frame["d"]
            .str.removeprefix("d_")
            .astype("int32")
        )
    )
    .sort_values("day_index", kind="stable")
    .set_index("d")
)

expected_day_index = np.arange(1, 1942, dtype=np.int32)
observed_day_index = calendar_lookup["day_index"].to_numpy(dtype=np.int32)

assert np.array_equal(observed_day_index, expected_day_index), (
    "Calendar day labels are not numerically continuous from d_1 through d_1941."
)

assert calendar_lookup["date"].is_monotonic_increasing, (
    "Observed sales days are not chronologically ordered after numeric day sorting."
)

assert calendar_lookup["date"].is_unique, (
    "Observed sales-day dates are not unique."
)

print(
    "Calendar chronology verified:",
    calendar_lookup.iloc[0][["day_index", "date"]].to_dict(),
    "->",
    calendar_lookup.iloc[-1][["day_index", "date"]].to_dict(),
)


Calendar chronology verified: {'day_index': 1, 'date': Timestamp('2011-01-29 00:00:00')} -> {'day_index': 1941, 'date': Timestamp('2016-05-22 00:00:00')}


## Temporal split manifest

A split is represented by its forecast origin $o$, context interval, and target interval.

For each origin:

$$
\text{context}(o)
=
[d_{o-511},d_o]
$$

and

$$
\text{target}(o)
=
[d_{o+1},d_{o+28}].
$$

The four training origins yield four deterministic windows per series, giving:

$$
30{,}490 \times 4
=
121{,}960
$$

training windows before batching.


In [7]:
CONTEXT_LENGTH = 512
FORECAST_HORIZON = 28

TRAIN_ORIGINS = [512, 960, 1409, 1857]
VALIDATION_ORIGIN = 1885
TEST_ORIGIN = 1913


def day_label(day_index: int) -> str:
    return f"d_{day_index}"


def date_for_day(day_index: int) -> str:
    """Return the ISO calendar date associated with an audited M5 day index."""
    assert 1 <= day_index <= 1941, (
        f"M5 day index must be within 1..1941; received {day_index}."
    )
    return calendar_lookup.loc[
        day_label(day_index),
        "date",
    ].date().isoformat()


def build_origin_record(
    split_name: str,
    origin: int,
    role: str,
) -> dict[str, Any]:
    context_start = origin - CONTEXT_LENGTH + 1
    context_end = origin
    target_start = origin + 1
    target_end = origin + FORECAST_HORIZON

    assert context_start >= 1
    assert target_end <= 1941

    return {
        "split_name": split_name,
        "role": role,
        "origin_day": day_label(origin),
        "origin_index": origin,
        "origin_date": date_for_day(origin),
        "context_start_day": day_label(context_start),
        "context_end_day": day_label(context_end),
        "context_start_date": date_for_day(context_start),
        "context_end_date": date_for_day(context_end),
        "target_start_day": day_label(target_start),
        "target_end_day": day_label(target_end),
        "target_start_date": date_for_day(target_start),
        "target_end_date": date_for_day(target_end),
        "context_length": CONTEXT_LENGTH,
        "forecast_horizon": FORECAST_HORIZON,
    }


split_records = [
    build_origin_record(
        split_name=f"train_origin_{origin}",
        origin=origin,
        role="train",
    )
    for origin in TRAIN_ORIGINS
]

split_records.append(
    build_origin_record(
        split_name="validation",
        origin=VALIDATION_ORIGIN,
        role="model_selection",
    )
)

split_records.append(
    build_origin_record(
        split_name="final_test",
        origin=TEST_ORIGIN,
        role="locked_final_evaluation",
    )
)

split_table = pd.DataFrame(split_records)

assert split_table.loc[
    split_table["split_name"] == "validation",
    "target_start_day",
].iloc[0] == "d_1886"

assert split_table.loc[
    split_table["split_name"] == "validation",
    "target_end_day",
].iloc[0] == "d_1913"

assert split_table.loc[
    split_table["split_name"] == "final_test",
    "target_start_day",
].iloc[0] == "d_1914"

assert split_table.loc[
    split_table["split_name"] == "final_test",
    "target_end_day",
].iloc[0] == "d_1941"

split_table


,split_name,role,origin_day,origin_index,origin_date,context_start_day,context_end_day,context_start_date,context_end_date,target_start_day,target_end_day,target_start_date,target_end_date,context_length,forecast_horizon
0,train_origin_512,train,d_512,512,2012-06-23,d_1,d_512,2011-01-29,2012-06-23,d_513,d_540,2012-06-24,2012-07-21,512,28
1,train_origin_960,train,d_960,960,2013-09-14,d_449,d_960,2012-04-21,2013-09-14,d_961,d_988,2013-09-15,2013-10-12,512,28
2,train_origin_1409,train,d_1409,1409,2014-12-07,d_898,d_1409,2013-07-14,2014-12-07,d_1410,d_1437,2014-12-08,2015-01-04,512,28
3,train_origin_1857,train,d_1857,1857,2016-02-28,d_1346,d_1857,2014-10-05,2016-02-28,d_1858,d_1885,2016-02-29,2016-03-27,512,28
4,validation,model_selection,d_1885,1885,2016-03-27,d_1374,d_1885,2014-11-02,2016-03-27,d_1886,d_1913,2016-03-28,2016-04-24,512,28
5,final_test,locked_final_evaluation,d_1913,1913,2016-04-24,d_1402,d_1913,2014-11-30,2016-04-24,d_1914,d_1941,2016-04-25,2016-05-22,512,28


## Leakage boundary

The development information boundary is fixed at $d_{1913}$.

During model development:

$$
\mathcal{D}_{dev}
\subseteq
\{d_1,\ldots,d_{1913}\}.
$$

The interval

$$
\{d_{1914},\ldots,d_{1941}\}
$$

is assigned exclusively to the final locked evaluation. Phase 02 records its existence and date range, but subsequent development notebooks must not use its target values for model or hyperparameter selection.


In [8]:
DEVELOPMENT_MAX_DAY = 1913
LOCKED_TEST_START_DAY = 1914
LOCKED_TEST_END_DAY = 1941

assert max(TRAIN_ORIGINS) + FORECAST_HORIZON <= VALIDATION_ORIGIN
assert VALIDATION_ORIGIN + FORECAST_HORIZON == DEVELOPMENT_MAX_DAY
assert TEST_ORIGIN + 1 == LOCKED_TEST_START_DAY
assert TEST_ORIGIN + FORECAST_HORIZON == LOCKED_TEST_END_DAY


## Retail covariate availability audit

The approved EA-TTM architecture uses future-known retail information. Phase 02 audits raw availability only; it does not yet choose the final encoded feature vector.

The audit records coverage for:

- event fields;
- state-specific SNAP indicators;
- weekly selling price;
- calendar identifiers required to derive horizon-specific calendar features.

Price missingness is retained as an observed property of M5 rather than silently imputed in this phase.


In [9]:
calendar_covariate_audit = {
    "event_name_1_non_null": int(calendar["event_name_1"].notna().sum()),
    "event_name_2_non_null": int(calendar["event_name_2"].notna().sum()),
    "snap_CA_unique": sorted(calendar["snap_CA"].dropna().unique().tolist()),
    "snap_TX_unique": sorted(calendar["snap_TX"].dropna().unique().tolist()),
    "snap_WI_unique": sorted(calendar["snap_WI"].dropna().unique().tolist()),
    "observed_sales_day_coverage": int(
        calendar["d"].isin(EXPECTED_DAY_COLUMNS).sum()
    ),
}

price_audit = {
    "rows": int(prices.shape[0]),
    "unique_store_item_pairs": int(
        prices[["store_id", "item_id"]].drop_duplicates().shape[0]
    ),
    "missing_sell_price_rows": int(prices["sell_price"].isna().sum()),
    "min_sell_price": float(prices["sell_price"].min()),
    "max_sell_price": float(prices["sell_price"].max()),
}

calendar_covariate_audit, price_audit


({'event_name_1_non_null': 162,
  'event_name_2_non_null': 5,
  'snap_CA_unique': [0, 1],
  'snap_TX_unique': [0, 1],
  'snap_WI_unique': [0, 1],
  'observed_sales_day_coverage': 1941},
 {'rows': 6841121,
  'unique_store_item_pairs': 30490,
  'missing_sell_price_rows': 0,
  'min_sell_price': 0.009999999776482582,
  'max_sell_price': 107.31999969482422})

## Training-history distribution audit

The development training-history distribution is summarized over:

$$
\{d_1,\ldots,d_{1885}\}.
$$

This is the complete historical region preceding the 28-day validation target. The summary is descriptive only; the formal extreme-label definition remains deferred to the dedicated regime-shift analysis phase.


In [10]:
TRAIN_HISTORY_END = VALIDATION_ORIGIN

training_history_columns = [
    day_label(day)
    for day in range(1, TRAIN_HISTORY_END + 1)
]

training_values = sales[training_history_columns].to_numpy(
    dtype=np.float32,
    copy=False,
)

training_observation_count = int(training_values.size)
training_zero_count = int(np.count_nonzero(training_values == 0.0))

training_distribution_summary = {
    "observation_count": training_observation_count,
    "zero_count": training_zero_count,
    "zero_rate": float(training_zero_count / training_observation_count),
    "mean": float(training_values.mean(dtype=np.float64)),
    "std": float(training_values.std(dtype=np.float64)),
    "min": float(training_values.min()),
    "max": float(training_values.max()),
}

training_distribution_summary


{'observation_count': 57473650,
 'zero_count': 39296634,
 'zero_rate': 0.6837330498411011,
 'mean': 1.1224584309505312,
 'std': 3.876981929807491,
 'min': 0.0,
 'max': 763.0}

## Series-level audit artifacts

Per-series descriptors are computed over the development training history. These descriptors support later stratification and regime-shift diagnostics without repeatedly scanning the full wide sales matrix.


In [11]:
series_audit = sales[SALES_IDENTIFIER_COLUMNS].copy()

series_audit["training_mean"] = training_values.mean(axis=1, dtype=np.float64)
series_audit["training_std"] = training_values.std(axis=1, dtype=np.float64)
series_audit["training_zero_rate"] = (
    np.count_nonzero(training_values == 0.0, axis=1)
    / training_values.shape[1]
)
series_audit["training_max"] = training_values.max(axis=1)
series_audit["training_total"] = training_values.sum(axis=1, dtype=np.float64)

daily_total_sales = pd.DataFrame(
    {
        "d": training_history_columns,
        "total_sales": training_values.sum(axis=0, dtype=np.float64),
    }
)

daily_total_sales["date"] = daily_total_sales["d"].map(
    calendar_lookup["date"]
)


## Efficient processed storage

Validated source tables are converted to Parquet. The Parquet artifacts are convenience representations; the raw CSV files and their SHA-256 fingerprints remain the immutable source of truth.


In [12]:
PROCESSED_PATHS = {
    "sales": PROCESSED_M5_DIR / "sales_train_evaluation.parquet",
    "calendar": PROCESSED_M5_DIR / "calendar.parquet",
    "prices": PROCESSED_M5_DIR / "sell_prices.parquet",
    "series_audit": PROCESSED_M5_DIR / "series_audit_summary.parquet",
    "daily_total_sales": PROCESSED_M5_DIR / "daily_total_sales.parquet",
}

sales.to_parquet(
    PROCESSED_PATHS["sales"],
    index=False,
    engine="pyarrow",
    compression="snappy",
)

calendar.to_parquet(
    PROCESSED_PATHS["calendar"],
    index=False,
    engine="pyarrow",
    compression="snappy",
)

prices.to_parquet(
    PROCESSED_PATHS["prices"],
    index=False,
    engine="pyarrow",
    compression="snappy",
)

series_audit.to_parquet(
    PROCESSED_PATHS["series_audit"],
    index=False,
    engine="pyarrow",
    compression="snappy",
)

daily_total_sales.to_parquet(
    PROCESSED_PATHS["daily_total_sales"],
    index=False,
    engine="pyarrow",
    compression="snappy",
)

assert all(path.exists() for path in PROCESSED_PATHS.values())


## Processed-artifact verification

The Parquet sales artifact is re-opened by metadata and by a minimal column read. This verifies persistence without duplicating the full in-memory table.


In [13]:
sales_parquet_file = pq.ParquetFile(PROCESSED_PATHS["sales"])

assert sales_parquet_file.metadata.num_rows == 30_490
assert sales_parquet_file.metadata.num_columns == len(sales_header)

parquet_probe = pd.read_parquet(
    PROCESSED_PATHS["sales"],
    columns=["id", "d_1", "d_1941"],
)

assert parquet_probe.shape == (30_490, 3)
assert parquet_probe["id"].equals(sales["id"])


## Persistent metadata

Four independent metadata artifacts are written:

1. **data manifest** — source paths, sizes, and raw-file hashes;
2. **data audit** — schema, dimensions, distribution summary, and covariate coverage;
3. **split manifest** — exact chronological origins and date ranges;
4. **run summary** — compact completion record used by the next phase.


In [14]:
DATA_MANIFEST_PATH = METADATA_DIR / "02_DATA_MANIFEST.json"
DATA_AUDIT_PATH = METADATA_DIR / "02_DATA_AUDIT.json"
SPLIT_MANIFEST_PATH = METADATA_DIR / "02_SPLIT_MANIFEST.json"
RUN_SUMMARY_PATH = METADATA_DIR / "02_RUN_SUMMARY.json"
SPLIT_TABLE_PATH = RESULTS_DIR / "02_SPLIT_TABLE.csv"

split_table.to_csv(SPLIT_TABLE_PATH, index=False)

data_manifest = {
    "project": "EA_TTM_RETAIL",
    "dataset": "M5 Forecasting - Accuracy",
    "created_utc": datetime.now(timezone.utc).isoformat(),
    "raw_files": raw_file_manifest,
    "dataset_sha256": dataset_sha256,
    "processed_artifacts": {
        key: str(path)
        for key, path in PROCESSED_PATHS.items()
    },
}

data_audit = {
    "sales_shape": list(sales.shape),
    "calendar_shape": list(calendar.shape),
    "prices_shape": list(prices.shape),
    "series_count": int(sales.shape[0]),
    "first_sales_day": EXPECTED_DAY_COLUMNS[0],
    "last_sales_day": EXPECTED_DAY_COLUMNS[-1],
    "training_history_end_day": day_label(TRAIN_HISTORY_END),
    "training_distribution": training_distribution_summary,
    "calendar_covariates": calendar_covariate_audit,
    "prices": price_audit,
}

split_manifest = {
    "context_length": CONTEXT_LENGTH,
    "forecast_horizon": FORECAST_HORIZON,
    "train_origins": TRAIN_ORIGINS,
    "validation_origin": VALIDATION_ORIGIN,
    "test_origin": TEST_ORIGIN,
    "training_window_count": int(sales.shape[0] * len(TRAIN_ORIGINS)),
    "validation_window_count": int(sales.shape[0]),
    "test_window_count": int(sales.shape[0]),
    "development_max_day": day_label(DEVELOPMENT_MAX_DAY),
    "locked_test_interval": [
        day_label(LOCKED_TEST_START_DAY),
        day_label(LOCKED_TEST_END_DAY),
    ],
    "splits": split_records,
}

split_payload = json.dumps(
    split_manifest,
    sort_keys=True,
    separators=(",", ":"),
).encode("utf-8")

split_sha256 = hashlib.sha256(split_payload).hexdigest()
split_manifest["split_sha256"] = split_sha256

run_summary = {
    "status": "PHASE_02_COMPLETE",
    "dataset_sha256": dataset_sha256,
    "split_sha256": split_sha256,
    "series_count": int(sales.shape[0]),
    "training_window_count": int(sales.shape[0] * len(TRAIN_ORIGINS)),
    "validation_window_count": int(sales.shape[0]),
    "test_window_count": int(sales.shape[0]),
    "locked_test_start": "d_1914",
    "locked_test_end": "d_1941",
}

for path, payload in (
    (DATA_MANIFEST_PATH, data_manifest),
    (DATA_AUDIT_PATH, data_audit),
    (SPLIT_MANIFEST_PATH, split_manifest),
    (RUN_SUMMARY_PATH, run_summary),
):
    path.write_text(
        json.dumps(payload, indent=2),
        encoding="utf-8",
    )


In [15]:
status_path = PROJECT_ROOT / "PROJECT_STATUS.md"

status_text = f"""# EA-TTM Project Status

## Current stage
Phase 02 — M5 data contract and temporal split audit completed.

## Dataset
- Dataset: M5 Forecasting - Accuracy
- Bottom-level series: {sales.shape[0]:,}
- Observed sales days: d_1 through d_1941
- Dataset SHA-256: `{dataset_sha256}`

## TTM contract carried forward
- Context length: 512
- Effective horizon: 28
- TTM revision: 512-48-ft-r2.1
- Decoder hidden width: 128
- Decoder patch/token count observed in Phase 01: 9

## Temporal protocol
- Training origins: {TRAIN_ORIGINS}
- Validation target: d_1886 through d_1913
- Locked final test: d_1914 through d_1941
- Split SHA-256: `{split_sha256}`

## Training windows
- Train: {sales.shape[0] * len(TRAIN_ORIGINS):,}
- Validation: {sales.shape[0]:,}
- Final test: {sales.shape[0]:,}

## Next phase
Establish the zero-shot retail baseline and distribution-shift evidence using the
validated M5 contract without using locked final-test targets for model selection.
"""

status_path.write_text(status_text, encoding="utf-8")


998

## Phase validation

Phase 02 passes only if:

$$
\begin{aligned}
&\text{all three raw files exist and are fingerprinted}
\\
&\land\ \text{30,490 unique bottom-level series are present}
\\
&\land\ d_1{:}d_{1941}\text{ is complete}
\\
&\land\ \text{calendar mapping is one-to-one}
\\
&\land\ \text{validation is }d_{1886}{:}d_{1913}
\\
&\land\ \text{locked test is }d_{1914}{:}d_{1941}
\\
&\land\ \text{processed Parquet artifacts reload successfully}.
\end{aligned}
$$


In [16]:
phase_validation = {
    "raw_files_present": all(path.exists() for path in REQUIRED_RAW_FILES.values()),
    "series_count_valid": int(sales.shape[0]) == 30_490,
    "sales_day_contract_valid": observed_day_columns == EXPECTED_DAY_COLUMNS,
    "calendar_mapping_valid": len(duplicated_required_calendar_days) == 0,
    "validation_interval_valid": (
        split_table.loc[
            split_table["split_name"] == "validation",
            ["target_start_day", "target_end_day"],
        ].iloc[0].tolist()
        == ["d_1886", "d_1913"]
    ),
    "test_interval_valid": (
        split_table.loc[
            split_table["split_name"] == "final_test",
            ["target_start_day", "target_end_day"],
        ].iloc[0].tolist()
        == ["d_1914", "d_1941"]
    ),
    "processed_artifacts_valid": all(
        path.exists()
        for path in PROCESSED_PATHS.values()
    ),
}

assert all(phase_validation.values()), (
    f"Phase 02 validation failed: {phase_validation}"
)

phase_validation


{'raw_files_present': True,
 'series_count_valid': True,
 'sales_day_contract_valid': True,
 'calendar_mapping_valid': True,
 'validation_interval_valid': True,
 'test_interval_valid': True,
 'processed_artifacts_valid': True}

## Return block

The return block contains the dataset identity, split identity, verified dimensions, and descriptive training-history statistics required to authorize the next phase.


In [17]:
return_block = {
    "status": "PHASE_02_COMPLETE",
    "dataset": "M5 Forecasting - Accuracy",
    "dataset_sha256": dataset_sha256,
    "sales_shape": list(sales.shape),
    "calendar_shape": list(calendar.shape),
    "prices_shape": list(prices.shape),
    "series_count": int(sales.shape[0]),
    "observed_sales_days": ["d_1", "d_1941"],
    "context_length": CONTEXT_LENGTH,
    "forecast_horizon": FORECAST_HORIZON,
    "train_origins": TRAIN_ORIGINS,
    "validation_origin": VALIDATION_ORIGIN,
    "validation_target": ["d_1886", "d_1913"],
    "test_origin": TEST_ORIGIN,
    "locked_test_target": ["d_1914", "d_1941"],
    "training_window_count": int(sales.shape[0] * len(TRAIN_ORIGINS)),
    "validation_window_count": int(sales.shape[0]),
    "test_window_count": int(sales.shape[0]),
    "training_distribution": training_distribution_summary,
    "calendar_covariate_audit": calendar_covariate_audit,
    "price_audit": price_audit,
    "split_sha256": split_sha256,
    "artifacts": {
        "data_manifest": str(DATA_MANIFEST_PATH),
        "data_audit": str(DATA_AUDIT_PATH),
        "split_manifest": str(SPLIT_MANIFEST_PATH),
        "run_summary": str(RUN_SUMMARY_PATH),
        "split_table": str(SPLIT_TABLE_PATH),
        "processed_sales": str(PROCESSED_PATHS["sales"]),
        "processed_calendar": str(PROCESSED_PATHS["calendar"]),
        "processed_prices": str(PROCESSED_PATHS["prices"]),
        "series_audit": str(PROCESSED_PATHS["series_audit"]),
        "daily_total_sales": str(PROCESSED_PATHS["daily_total_sales"]),
        "project_status": str(status_path),
    },
}

print("PHASE 02 RETURN BLOCK")
print(json.dumps(return_block, indent=2))


PHASE 02 RETURN BLOCK
{
  "status": "PHASE_02_COMPLETE",
  "dataset": "M5 Forecasting - Accuracy",
  "dataset_sha256": "83962ce8bbcb0757215d045f81cac3a7cf4465943334ffa168af326bedc5ed59",
  "sales_shape": [
    30490,
    1947
  ],
  "calendar_shape": [
    1969,
    14
  ],
  "prices_shape": [
    6841121,
    4
  ],
  "series_count": 30490,
  "observed_sales_days": [
    "d_1",
    "d_1941"
  ],
  "context_length": 512,
  "forecast_horizon": 28,
  "train_origins": [
    512,
    960,
    1409,
    1857
  ],
  "validation_origin": 1885,
  "validation_target": [
    "d_1886",
    "d_1913"
  ],
  "test_origin": 1913,
  "locked_test_target": [
    "d_1914",
    "d_1941"
  ],
  "training_window_count": 121960,
  "validation_window_count": 30490,
  "test_window_count": 30490,
  "training_distribution": {
    "observation_count": 57473650,
    "zero_count": 39296634,
    "zero_rate": 0.6837330498411011,
    "mean": 1.1224584309505312,
    "std": 3.876981929807491,
    "min": 0.0,
    "max": 